In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Permite importar tu paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))


# 07 · Validación e interpretación — abrir la prueba, una sola vez

> **Esqueleto.** Guía completa: `curso/kit-estudiante/guia-validacion-registro-y-app.md`, secciones 1 a 4.

Hasta aquí **nadie ha mirado el conjunto de prueba**. Hoy se abre, y las reglas son estrictas:

1. Todas las decisiones (modelo, parámetros, calibración) se toman **antes** y se **escriben**; solo con entrenamiento.
2. La prueba se evalúa **una vez**. Si algo no gusta, no se "ajusta y se vuelve a mirar".
3. No basta un número: se pide el **intervalo** (remuestreando **entidades**), la **comparación con el baseline**,
   el **error por segmento** y **por qué** decide así el modelo.

## 0. Cargar los datos de entrenamiento

In [ ]:
import joblib
import pandas as pd

# === Configuración: completa estas dos líneas (las mismas de tus notebooks anteriores) ===
ID_COL = None     # TODO: columna que identifica a la entidad que se repite (None si no hay entidades repetidas)
TARGET = None     # TODO: tu columna objetivo

train_path = Path.cwd().resolve().parent / "data" / "processed" / "train.joblib"
if not train_path.exists():
    train = None
    print("Todavía no hay datos de entrenamiento: completa y ejecuta antes el notebook 03_preprocesamiento.")
else:
    train = joblib.load(train_path)
    print(f"entrenamiento: {train.shape[0]:,} filas x {train.shape[1]} columnas")

listo = train is not None and TARGET is not None
if train is not None and TARGET is None:
    print("Define TARGET (y, si hay entidades repetidas, ID_COL) arriba para continuar.")

## 1. Decisiones congeladas — **antes** de abrir la prueba

Escribe aquí lo que decidiste con **validación**, no con la prueba. Es lo que defenderás ante el instructor:
si cambias una decisión **después** de ver la prueba, lo declaras en el reporte.

In [ ]:
DECISIONES = {
    "modelo_final": None,         # TODO: nombre del modelo elegido (p. ej. "random forest")
    "parametros": None,           # TODO: dict de parámetros optimizados, o "partida" si la optimización no mejoró de forma distinguible
    "calibrar": None,             # TODO: True / False, decidido con entidades reservadas (no con la prueba)
    "metrica_principal": None,    # TODO: la métrica de negocio con la que elegiste
    "regla_de_decision": None,    # TODO: p. ej. "clase más probable" o "menor costo esperado"
    "decidido_el": None,          # TODO: fecha y hora en que lo congelaron, p. ej. "2026-11-05 09:10"
}
faltan = [clave for clave, valor in DECISIONES.items() if valor is None]
decisiones_completas = not faltan
if faltan:
    print("Antes de abrir la prueba completa estas decisiones:", ", ".join(faltan))
else:
    print("Decisiones congeladas. Cópialas a reports/07_validacion.md antes de seguir.")

## 2. ¿Hay que calibrar las probabilidades?

Si usas la **probabilidad** para decidir (un umbral, un costo), comprueba su calibración **con entidades reservadas**
del entrenamiento —las mismas de la optimización—, **no con la prueba**. Compara la calidad de las probabilidades
(log loss, curva de calibración) con y sin calibrar, y decide.

> Si no hace falta (tu decisión es la clase más probable), escribe `calibrar = False` arriba y di por qué en el reporte.

In [ ]:
# TODO: separa un 25 % de las ENTIDADES (no de las filas) dentro del entrenamiento, ajusta el modelo con el 75 %,
#       compara probabilidades sin calibrar y calibradas en el 25 %, y decide `calibrar`.
#       Pista: sklearn.calibration.CalibratedClassifierCV y calibration_curve; la guía, sección 2.
print("Pendiente: decisión de calibración (con entidades reservadas, nunca con la prueba).")

## 3. Entrenar el modelo final con **todo** el entrenamiento

Con las decisiones ya tomadas, el modelo final se entrena con el 100 % del conjunto de entrenamiento y se guarda
**entero** (preprocesamiento + modelo) para registrarlo en el notebook 08.

In [ ]:
try:
    from proyecto.models.train import build_pipeline   # TODO: la creaste en la etapa del torneo
except ModuleNotFoundError:
    build_pipeline = None
    print("TODO: necesitas src/proyecto/models/train.py con build_pipeline(modelo). Vuelve al notebook 05.")

modelo_final = None
if listo and decisiones_completas and build_pipeline is not None:
    pass
    # TODO: modelo = ... (según DECISIONES["modelo_final"] y ["parametros"])
    # TODO: modelo_final = build_pipeline(modelo).fit(X_train, y_train)   (con calibración si DECISIONES["calibrar"])
    # TODO: guarda: joblib.dump(modelo_final, Path.cwd().resolve().parent / "models" / "final_pipeline.joblib")
else:
    print("Pendiente: completa la configuración, las decisiones y build_pipeline.")

## 4. Abrir la prueba — **una sola vez**

A partir de aquí **no se cambia nada del modelo**. Esta sección deja una marca (`reports/prueba_abierta.txt`) para que
quede constancia de cuándo se abrió; el instructor la mirará.

In [ ]:
from datetime import datetime

marca = Path.cwd().resolve().parent / "reports" / "prueba_abierta.txt"
test_path = Path.cwd().resolve().parent / "data" / "processed" / "test.joblib"
test = None

if not (decisiones_completas and modelo_final is not None):
    print("La prueba sigue CERRADA: faltan las decisiones congeladas o el modelo final.")
elif not test_path.exists():
    print("No existe data/processed/test.joblib: se genera en el notebook 03_preprocesamiento.")
else:
    if marca.exists():
        print("AVISO:", marca.read_text(encoding="utf-8").strip())
        print("Si abres la prueba otra vez para decidir algo, deja de medir generalización: decláralo en el reporte.")
    else:
        marca.write_text(f"La prueba se abrió por primera vez el {datetime.now():%Y-%m-%d %H:%M}.", encoding="utf-8")
    test = joblib.load(test_path)
    print(f"prueba: {test.shape[0]:,} filas (primera y única evaluación)")

In [ ]:
# TODO: predice en la prueba con el modelo final y calcula tu métrica principal y las de control.
# TODO: calcula la misma métrica para el BASELINE (el del torneo) en la prueba.
metricas_prueba = None

## 5. Intervalo y comparación con el baseline — remuestreando **entidades**

Un número sin intervalo no dice cuánto confiar. Remuestrea **entidades completas** (no filas): las filas de una misma
entidad no son independientes. Esta función ya está lista; tú le pasas, **por fila de la prueba**, la pérdida de
tu modelo (por ejemplo el costo, o 0/1 de error) y la entidad de cada fila.

In [ ]:
import numpy as np


def bootstrap_por_entidad(valores, entidades, n_boot=1000, seed=42):
    """Promedio de `valores` con intervalo del 95 %, remuestreando entidades completas.

    Para comparar dos modelos, pasa la DIFERENCIA fila a fila de sus pérdidas (modelo A menos modelo B):
    si todo el intervalo queda por debajo de cero, A pierde menos con confianza del 95 %.
    """
    tabla = pd.DataFrame({"g": np.asarray(entidades), "v": np.asarray(valores, dtype=float)})
    por_entidad = tabla.groupby("g")["v"].agg(["sum", "count"])
    sumas, cuentas = por_entidad["sum"].to_numpy(), por_entidad["count"].to_numpy()
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(sumas), size=(n_boot, len(sumas)))
    remuestras = sumas[idx].sum(axis=1) / cuentas[idx].sum(axis=1)
    return float(sumas.sum() / cuentas.sum()), float(np.percentile(remuestras, 2.5)), float(np.percentile(remuestras, 97.5))


# Prueba de la función con datos inventados (no son tus datos): debe dar un promedio de 0,5 con un intervalo razonable
rng = np.random.default_rng(0)
print("ejemplo de juguete:", bootstrap_por_entidad(rng.integers(0, 2, 400), np.repeat(np.arange(50), 8)))

# TODO: llama a bootstrap_por_entidad con tu pérdida por fila de la prueba y con la diferencia frente al baseline.

## 6. Errores por segmento y carga de trabajo

Un buen promedio puede esconder un grupo al que el modelo le va mal. Elige 1 o 2 variables de **segmento** con
sentido para tu negocio (edad por tramos, región, tipo de cliente…) y mira el error en cada una.

Si tu modelo **prioriza revisión humana**, mide también **cuántos casos marca** como positivos frente a cuántos hay
realmente: más revisión es más trabajo, y el costo o la métrica no lo ven.

In [ ]:
# TODO: tabla por segmento: número de casos, tu métrica, % que el modelo marca como positivo y % real.
# TODO: carga de trabajo: compara lo que decide el modelo con lo real (proporciones).
print("Pendiente: errores por segmento y carga de trabajo.")

## 7. ¿Por qué decide así el modelo? (SHAP u otra explicación)

Los valores SHAP (o las importancias por permutación) muestran qué variables mueven la predicción. Recuerda el límite:
**explican al modelo, no al mundo**; no dicen si el modelo acertó. Y las variables correlacionadas se reparten el crédito.

In [ ]:
try:
    import shap   # noqa: F401
except ModuleNotFoundError:
    print("shap no está instalado: pídele a Claude la alternativa (sklearn.inspection.permutation_importance).")

# TODO: importancia global (qué variables pesan más) y dos clientes concretos (uno por cada resultado).
print("Pendiente: interpretación.")

## 8. Guardar los resultados de la etapa

Guarda las métricas de la prueba (con su intervalo) y tus decisiones en `reports/07_metricas_test.json`; el notebook 08
las registra junto al modelo. Y completa `reports/07_validacion.md`.

In [ ]:
import json

resultados = {"decisiones": DECISIONES, "prueba": metricas_prueba}   # TODO: añade el intervalo y el baseline
if decisiones_completas and metricas_prueba is not None:
    destino = Path.cwd().resolve().parent / "reports" / "07_metricas_test.json"
    destino.write_text(json.dumps(resultados, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    print("guardado:", destino.name)
else:
    print("Pendiente: calcula las métricas de la prueba (sección 4) antes de guardar.")